# FMP Equities

Read-first examples for FMP-backed equity data in Quant Warehouse. Set `RUN_REFRESH = True` only when you want to download missing data.

In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openbb import obb

for env_dir in [Path.cwd(), *Path.cwd().parents]:
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.openbb_fetch import SECTION_ROUTES
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.sections import (
    CRYPTO_PRICES_LIBRARY,
    CRYPTO_PRICES_SECTION,
    CURRENCY_PRICES_LIBRARY,
    CURRENCY_PRICES_SECTION,
    DEFAULT_CRYPTO_SYMBOLS,
    DEFAULT_CURRENCY_SYMBOLS,
    DEFAULT_ECONOMIC_SERIES,
    DEFAULT_INDEX_SYMBOLS,
    EQUITY_CALENDAR_SECTIONS,
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    ETF_PRICES_SECTION,
    FUND_PRICES_LIBRARY,
    FUND_PRICES_SECTION,
    INDEX_PRICES_LIBRARY,
    INDEX_PRICES_SECTION,
    MACRO_CALENDAR_LIBRARY,
    MACRO_CALENDAR_SECTION,
    MACRO_CALENDAR_SYMBOL,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_ECONOMIC_SECTION,
    MACRO_RISK_PREMIUM_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION,
    MACRO_TREASURY_LIBRARY,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_LIBRARY,
    MACRO_YIELD_CURVE_SECTION,
    RISK_PREMIUM_SYMBOL,
    TREASURY_BUNDLE_SYMBOL,
    YIELD_CURVE_BUNDLE_SYMBOL,
    fundamental_library,
)
from quant_warehouse.warehouse.storage import provider_library

configure_openbb_credentials()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()
PROVIDER = "fmp"
RUN_REFRESH = False

EQUITY_SYMBOL = "AAPL"

In [ ]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def route_table(section_to_base_library: dict[str, str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "section": section,
                "openbb_route": SECTION_ROUTES.get(section),
                "provider_library": provider_library(base_library, PROVIDER),
            }
            for section, base_library in section_to_base_library.items()
        ]
    )


def run_openbb_route(label: str, fn):
    try:
        result = fn()
        frame = result.to_df()
        return {
            "label": label,
            "provider": getattr(result, "provider", None),
            "rows": 0 if frame is None else len(frame),
            "columns": [] if frame is None else list(frame.columns),
            "error": None,
        }
    except Exception as exc:
        return {
            "label": label,
            "provider": PROVIDER,
            "rows": None,
            "columns": [],
            "error": str(exc),
        }

## Available Equity Sections

In [ ]:
equity_sections = ["prices", "profile", *FMP_HISTORICAL_EQUITY_SECTIONS, *FMP_EXTENDED_EQUITY_SECTIONS]
equity_route_libraries = {
    "prices": EQUITY_PRICES_LIBRARY,
    "profile": "profile",
    **{section: fundamental_library(section) for section in FMP_HISTORICAL_EQUITY_SECTIONS},
    **{section: fundamental_library(section) for section in FMP_EXTENDED_EQUITY_SECTIONS},
}
route_table(equity_route_libraries)

## Local Storage Coverage

In [ ]:
state_table(EQUITY_SYMBOL, equity_sections)

## Prices and Profile

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_prices(EQUITY_SYMBOL, providers=[PROVIDER])
    warehouse.refresh_profile(EQUITY_SYMBOL, provider=PROVIDER)

preview(warehouse.read_prices(EQUITY_SYMBOL, provider=PROVIDER))

In [ ]:
profile = warehouse.read_profile(EQUITY_SYMBOL, provider=PROVIDER)
asdict(profile) if profile is not None else None

## Core Fundamentals

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_fundamentals(
        EQUITY_SYMBOL,
        sections=["income", "balance", "cash", "metrics", "ratios"],
        providers=[PROVIDER],
        period="quarter",
    )

{
    "income": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="income", provider=PROVIDER)),
    "balance": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="balance", provider=PROVIDER)),
    "cash": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="cash", provider=PROVIDER)),
    "ratios": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="ratios", provider=PROVIDER)),
}

## Events, Estimates, Ownership, and Calendars

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_fundamentals(
        EQUITY_SYMBOL,
        sections=["dividends", "historical_splits", "filings", "estimates_price_target", "ownership_insider_trading"],
        providers=[PROVIDER],
    )

{
    "dividends": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="dividends", provider=PROVIDER)),
    "historical_splits": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="historical_splits", provider=PROVIDER)),
    "filings": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="filings", provider=PROVIDER)),
    "estimates_price_target": preview(warehouse.read_fundamentals(EQUITY_SYMBOL, section="estimates_price_target", provider=PROVIDER)),
}

In [ ]:
calendar_routes = {section: fundamental_library(section) for section in EQUITY_CALENDAR_SECTIONS}
route_table(calendar_routes)

In [ ]:
if RUN_REFRESH:
    warehouse.equity_calendar.refresh_all(provider=PROVIDER, start_date="2024-01-01")

state_table("EQUITY_CALENDAR", EQUITY_CALENDAR_SECTIONS)

<!-- quant-trader-eda -->
## Quant Trader EDA

Quick checks for a single equity: price risk, drawdowns, benchmark beta, and whether fundamental fields are populated enough to support factor work.

In [ ]:
def price_eda(frame: pd.DataFrame, annualization: int = 252) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame is None or frame.empty or "close" not in frame.columns:
        return pd.DataFrame(), pd.DataFrame()
    close = pd.to_numeric(frame["close"], errors="coerce").dropna()
    returns = close.pct_change().dropna()
    if returns.empty:
        return pd.DataFrame(), pd.DataFrame()
    equity_curve = (1 + returns).cumprod()
    drawdown = equity_curve / equity_curve.cummax() - 1
    summary = pd.DataFrame(
        {
            "observations": [int(len(returns))],
            "start": [close.index.min()],
            "end": [close.index.max()],
            "total_return": [close.iloc[-1] / close.iloc[0] - 1],
            "annualized_return": [(1 + returns.mean()) ** annualization - 1],
            "annualized_volatility": [returns.std() * annualization ** 0.5],
            "sharpe_0_rf": [returns.mean() / returns.std() * annualization ** 0.5 if returns.std() else None],
            "max_drawdown": [drawdown.min()],
            "best_day": [returns.max()],
            "worst_day": [returns.min()],
        }
    )
    diagnostics = pd.DataFrame(
        {
            "close": close,
            "return": returns.reindex(close.index),
            "rolling_21d_vol": returns.rolling(21).std() * annualization ** 0.5,
            "rolling_63d_vol": returns.rolling(63).std() * annualization ** 0.5,
            "drawdown": drawdown.reindex(close.index),
        }
    )
    return summary, diagnostics

In [ ]:
equity_prices = warehouse.read_prices(EQUITY_SYMBOL, provider=PROVIDER)
equity_summary, equity_diagnostics = price_eda(equity_prices)
equity_summary

In [ ]:
preview(equity_diagnostics[["close", "rolling_21d_vol", "rolling_63d_vol", "drawdown"]], rows=10)

In [ ]:
benchmark = warehouse.etf.read_prices("SPY", provider=PROVIDER)
if equity_prices.empty or benchmark.empty:
    beta_table = pd.DataFrame()
else:
    asset_returns = equity_prices["close"].pct_change().rename(EQUITY_SYMBOL)
    benchmark_returns = benchmark["close"].pct_change().rename("SPY")
    joined = pd.concat([asset_returns, benchmark_returns], axis=1).dropna()
    if joined.empty or joined["SPY"].var() == 0:
        beta_table = pd.DataFrame()
    else:
        beta = joined[EQUITY_SYMBOL].cov(joined["SPY"]) / joined["SPY"].var()
        beta_table = pd.DataFrame({
            "correlation_to_spy": [joined[EQUITY_SYMBOL].corr(joined["SPY"])],
            "beta_to_spy": [beta],
            "tracking_volatility": [(joined[EQUITY_SYMBOL] - joined["SPY"]).std() * 252 ** 0.5],
        })
beta_table

In [ ]:
fundamental_sections = ["income", "balance", "cash", "metrics", "ratios"]
coverage = state_table(EQUITY_SYMBOL, fundamental_sections)
coverage[[column for column in ["section", "row_count", "column_count", "min_date", "max_date"] if column in coverage.columns]]

In [ ]:
ratios = warehouse.read_fundamentals(EQUITY_SYMBOL, section="ratios", provider=PROVIDER)
if ratios.empty:
    pd.DataFrame()
else:
    candidate_columns = [
        "price_to_earnings_ratio", "price_to_book_ratio", "price_to_sales_ratio",
        "debt_to_equity_ratio", "return_on_equity", "net_profit_margin",
    ]
    ratios[[column for column in candidate_columns if column in ratios.columns]].tail(8)

<!-- quant-trader-signal-eda -->
## Signal Research for P&L

These cells frame the data as candidate trading signals. They are diagnostics for research, not a recommendation to trade the example symbol.

In [ ]:
def signal_backtest(close: pd.Series, signal: pd.Series, annualization: int = 252) -> pd.DataFrame:
    close = pd.to_numeric(close, errors="coerce").dropna()
    returns = close.pct_change().dropna()
    aligned_signal = signal.reindex(returns.index).shift(1).fillna(0).clip(-1, 1)
    strategy_returns = aligned_signal * returns
    if strategy_returns.empty:
        return pd.DataFrame()
    curve = (1 + strategy_returns).cumprod()
    buy_hold = (1 + returns).cumprod()
    drawdown = curve / curve.cummax() - 1
    turnover = aligned_signal.diff().abs().fillna(aligned_signal.abs())
    return pd.DataFrame({
        "strategy_total_return": [curve.iloc[-1] - 1],
        "buy_hold_total_return": [buy_hold.iloc[-1] - 1],
        "strategy_annualized_return": [(1 + strategy_returns.mean()) ** annualization - 1],
        "strategy_annualized_volatility": [strategy_returns.std() * annualization ** 0.5],
        "strategy_sharpe_0_rf": [strategy_returns.mean() / strategy_returns.std() * annualization ** 0.5 if strategy_returns.std() else None],
        "strategy_max_drawdown": [drawdown.min()],
        "hit_rate": [(strategy_returns > 0).mean()],
        "average_exposure": [aligned_signal.abs().mean()],
        "average_daily_turnover": [turnover.mean()],
        "latest_signal": [aligned_signal.iloc[-1]],
    })

In [ ]:
equity_prices = warehouse.read_prices(EQUITY_SYMBOL, provider=PROVIDER)
if equity_prices.empty or "close" not in equity_prices.columns:
    pd.DataFrame()
else:
    close = equity_prices["close"]
    trend_signal = (close > close.rolling(200).mean()).astype(float)
    signal_backtest(close, trend_signal)

In [ ]:
if equity_prices.empty or "close" not in equity_prices.columns:
    pd.DataFrame()
else:
    close = equity_prices["close"]
    returns = close.pct_change()
    vol_21 = returns.rolling(21).std() * 252 ** 0.5
    trend = close / close.rolling(200).mean() - 1
    breakout = close / close.rolling(63).max() - 1
    candidate = pd.DataFrame({
        "close": close,
        "trend_200d_distance": trend,
        "breakout_63d_distance": breakout,
        "realized_vol_21d": vol_21,
        "vol_target_weight_15pct_cap_1x": (0.15 / vol_21).clip(0, 1),
        "risk_state": pd.cut(vol_21, bins=[-float("inf"), 0.2, 0.4, float("inf")], labels=["normal", "elevated", "high"]),
    })
    preview(candidate, rows=15)

In [ ]:
ratios = warehouse.read_fundamentals(EQUITY_SYMBOL, section="ratios", provider=PROVIDER)
metrics = warehouse.read_fundamentals(EQUITY_SYMBOL, section="metrics", provider=PROVIDER)
frames = []
for name, frame in {"ratios": ratios, "metrics": metrics}.items():
    if not frame.empty:
        latest = frame.tail(1).copy()
        latest.index = [name]
        frames.append(latest)
if not frames:
    pd.DataFrame()
else:
    latest_fundamentals = pd.concat(frames, axis=0)
    signal_columns = [
        "price_to_earnings_ratio", "price_to_book_ratio", "price_to_sales_ratio",
        "return_on_equity", "net_profit_margin", "debt_to_equity_ratio",
        "free_cash_flow_yield", "earnings_yield",
    ]
    latest_fundamentals[[column for column in signal_columns if column in latest_fundamentals.columns]]

<!-- output-analysis -->
## Analysis Notes From This Run

For `AAPL`, the stored FMP history is strongly up over the full sample but has also experienced an 81.8% max drawdown, so raw long-only exposure has meaningful crash risk. The latest stored close is above the 200-day average by about 9.1%, and the 21-day realized volatility is about 22.5%, which is a constructive trend state but not a low-risk one.

Against `SPY`, the daily correlation is about 0.49 and the estimated beta/tracking behavior is materially idiosyncratic. For a quant trader, this makes `AAPL` more useful as a single-name alpha/risk sleeve than as a market proxy. The practical next checks are whether valuation/fundamental signals improve entries and whether volatility targeting reduces the large historical drawdown.